# Memory Compaction

> **Reduce storage and token cost through progressive summarization: from raw transcript to key points to one-liner to tags.**

Imagine you record every meeting at work, word for word. After a year you have thousands of pages. Most of those pages are filler: "Let me pull that up," "Sure, sounds good," "Sorry, you were saying?" The real decisions and action items fill maybe 10% of the text.

Now imagine you keep four versions of each meeting. The full transcript. A bullet-point summary. A one-liner. A set of tags. You'd scan the tags for quick searches. You'd read the one-liner for context. You'd only open the full transcript when exact wording matters.

Memory Compaction applies this idea to an agent's memory store. Raw memories (full conversation transcripts, detailed tool outputs, verbose reasoning traces) consume significant storage and token budget. Token budget is the cost of including text in a prompt. Compaction transforms verbose records into denser versions that keep essential information and discard filler.

This technique draws on Tiago Forte's Progressive Summarization method for personal knowledge management. Each memory exists at multiple fidelity levels. Level 0 (L0) is the raw transcript. Level 1 (L1) extracts key points. Level 2 (L2) distills those into a single sentence. Level 3 (L3) reduces everything to tags and entity references (structured labels like person names, dates, and project names).

The practical benefit is dramatic. A compacted memory store can hold 10x to 100x more semantic content in the same storage budget. Retrieval gets faster because compressed entries are smaller. The agent can choose the right level of detail for each situation.

**By the end of this notebook you'll understand:**
- How to compress memories through four progressive levels.
- How to build a compaction engine that uses LLM calls for each level.
- How to schedule compaction based on age and access patterns.
- How to retrieve memories at any fidelity level on demand.

## Key Concepts

- **Progressive Summarization Levels**: A hierarchy of compression. L0 is the raw text. L1 has key points. L2 is a one-line summary. L3 is tags and entity references only. Each level trades fidelity (completeness) for compactness (smaller size).
- **Lossy vs. Lossless Compression**: Lossless techniques (like extracting structured fields from unstructured text) keep all information. Lossy techniques (like summarization) intentionally discard detail. Compaction uses both strategically.
- **Detail Levels on Retrieval**: The ability to retrieve a memory at any compaction level. Quick scans use L2 or L3. Deep dives request L0 or L1. This lets the agent budget context tokens precisely.
- **Compaction Scheduling**: Policies for when compression happens. Options include age-based (compress after N hours), access-based (compress after M turns without access), and storage-pressure-based (compress when the store exceeds capacity).
- **Reversibility**: Keeping the original L0 in cold storage (cheap, slower storage) so compaction is non-destructive. The agent can always "zoom in" to the raw memory if a summary proves insufficient.
- **Storage-Fidelity Tradeoff**: The engineering decision about how much detail to retain at each level. You're balancing storage cost against information loss.
- **Entity Extraction**: Pulling structured entities (people, dates, projects, decisions) from raw text as part of the compaction process. This creates a queryable index alongside the compressed memories.

## Architecture

<p align="center">
  <img src="../../images/diagrams/15_memory_compaction.svg" alt="diagram" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    RM["Raw Memory\n(L0: Full fidelity)"] --> CE["Compaction\nEngine"]

    CE --> L1["L1: Key Points\n(~30% of original)"]
    L1 --> CE2["Compaction\nEngine"]
    CE2 --> L2["L2: One-liner\n(~5% of original)"]
    L2 --> CE3["Compaction\nEngine"]
    CE3 --> L3["L3: Tags Only\n(~1% of original)"]

    RM -->|"Archive original"| CS["Cold Storage"]

    RQ["Retrieval Query"] --> LS["Level Selector"]
    LS -->|"Quick scan"| L3
    LS -->|"Summary"| L2
    LS -->|"Moderate detail"| L1
    LS -->|"Full detail"| CS

    style RM fill:#2d4a7a,stroke:#49a,color:#fff
    style L1 fill:#2d5a5a,stroke:#4aa,color:#fff
    style L2 fill:#5a5a2d,stroke:#aa4,color:#fff
    style L3 fill:#5a2d2d,stroke:#a44,color:#fff
```

</details>

**Data flow**: A raw memory (L0) enters the system at full fidelity. The compaction engine progressively compresses it through three stages. L1 extracts key points (roughly 30% of the original tokens). L2 distills to a single-sentence summary (roughly 5%). L3 keeps only tags and entity references (roughly 1%).

The original L0 is archived in cold storage so you can always recover full detail. When a retrieval query arrives, the level selector picks the right detail level. It considers the query's needs and the available context budget.

## Setup

Install dependencies and configure API access. You'll need an OpenAI API key stored in a `.env` file or as an environment variable.

In [ ]:
%pip install -q openai python-dotenv

Import the OpenAI SDK and standard library helpers. The API key loads from a `.env` file.

In [ ]:
import os
import json
import time
import uuid
import re
from enum import IntEnum
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # reads OPENAI_API_KEY from .env

client = OpenAI()  # picks up OPENAI_API_KEY automatically
MODEL = "gpt-4o-mini"

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file" 

## Implementation

We'll build four components:

1. **CompactionLevel** and **CompactedMemory**: data structures that hold content at multiple fidelity levels.
2. **CompactionEngine**: uses LLM calls to compress from one level to the next.
3. **MemoryStore**: stores memories and supports level-aware retrieval.
4. **CompactionScheduler**: decides which memories are due for further compression.

### Data Structures

Think of a filing system with four drawers. The top drawer holds full documents. The second holds bullet-point summaries clipped to each document. The third holds sticky-note synopses. The bottom drawer holds index cards with only keywords. Each drawer gives you a different speed-vs-detail tradeoff.

`CompactionLevel` is an integer enum that labels the four drawers. `CompactedMemory` holds the content for whichever levels have been computed so far.

In [ ]:
class CompactionLevel(IntEnum):
    """Four progressive compression levels."""
    L0_RAW = 0          # Full original text
    L1_KEY_POINTS = 1   # Bullet-point extraction (~30% of L0)
    L2_SUMMARY = 2      # Single-sentence distillation (~5% of L0)
    L3_TAGS = 3         # Tags and entity labels only (~1% of L0)


@dataclass
class CompactedMemory:
    """A memory stored at one or more compaction levels."""
    memory_id: str
    levels: dict          # CompactionLevel -> str content
    entities: list        # Extracted entity names
    created_at: float     # Unix timestamp
    last_accessed: float  # Unix timestamp
    access_count: int = 0
    current_level: int = 0  # Highest (most compressed) level computed

    def char_count_at(self, level: int) -> int:
        """Return character count at a given level, or 0 if not computed."""
        content = self.levels.get(level, "")
        return len(content)

### Compaction Engine

The engine does the actual compression. Each transition (L0 to L1, L1 to L2, L2 to L3) uses a different LLM prompt tailored to the target format.

- **L0 to L1**: Extract 3 to 5 key points as bullet items.
- **L1 to L2**: Distill those bullets into a single sentence.
- **L2 to L3**: Extract only entity names and topic tags as a JSON object.

Each step is a separate LLM call. This keeps prompts focused and outputs predictable.

In [ ]:
class CompactionEngine:
    """Compresses memories from one level to the next using LLM calls."""

    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini"):
        self.client = client
        self.model = model

    def _call_llm(self, system_prompt: str, user_content: str) -> str:
        """Send a single chat completion request."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content},
            ],
            temperature=0.0,
        )
        return response.choices[0].message.content

    def compact_to_l1(self, raw_text: str) -> str:
        """L0 -> L1: Extract 3-5 key points as bullet items."""
        prompt = (
            "Extract the 3 to 5 most important points from this text. "
            "Return only bullet points, one per line, starting with '- '. "
            "Keep each point to one sentence."
        )
        return self._call_llm(prompt, raw_text)

    def compact_to_l2(self, key_points: str) -> str:
        """L1 -> L2: Distill key points into a single sentence."""
        prompt = (
            "Distill these key points into a single sentence of at most "
            "25 words. Return only the sentence, no extra text."
        )
        return self._call_llm(prompt, key_points)

    def compact_to_l3(self, summary: str) -> tuple:
        """L2 -> L3: Extract tags and entities as structured data.

        Returns (tag_string, entity_list).
        """
        prompt = (
            "Extract entity tags from this text. Return a JSON object with "
            "two keys: \"tags\" (topic keywords as a list of strings) and "
            "\"entities\" (proper nouns like people, projects, tools as a "
            "list of strings). Return only valid JSON, no markdown fences."
        )
        raw = self._call_llm(prompt, summary)
        # Strip markdown code fences if the model wraps the JSON
        cleaned = re.sub(r"^```(?:json)?\n?|```$", "", raw.strip())
        parsed = json.loads(cleaned)
        tag_str = ", ".join(parsed.get("tags", []))
        entities = parsed.get("entities", [])
        return tag_str, entities

The `compact` method orchestrates progressive compression. It walks from the memory's current level up to the target level, calling the right method at each step. For example, compacting from L0 to L3 runs L0 to L1, then L1 to L2, then L2 to L3 in sequence.

In [ ]:
    def compact(self, memory: CompactedMemory, target_level: int) -> CompactedMemory:
        """Progressively compact a memory up to the target level.

        Runs each intermediate step if needed. For example, compacting
        from L0 to L3 runs L0->L1, L1->L2, and L2->L3 in sequence.
        """
        current = memory.current_level
        while current < target_level:
            next_level = current + 1
            source_text = memory.levels[current]

            if next_level == CompactionLevel.L1_KEY_POINTS:
                memory.levels[next_level] = self.compact_to_l1(source_text)
            elif next_level == CompactionLevel.L2_SUMMARY:
                memory.levels[next_level] = self.compact_to_l2(source_text)
            elif next_level == CompactionLevel.L3_TAGS:
                tag_str, entities = self.compact_to_l3(source_text)
                memory.levels[next_level] = tag_str
                memory.entities = entities

            memory.current_level = next_level
            current = next_level

        return memory

### Memory Store

The store holds all memories and their compaction levels. It also manages cold storage: when a memory gets compacted past L0, the original raw text moves to a separate archive. This keeps the active store small while preserving full-fidelity access on demand.

In [ ]:
class MemoryStore:
    """Stores memories at multiple compaction levels with cold storage."""

    def __init__(self, engine: CompactionEngine):
        self.engine = engine
        self.memories: dict[str, CompactedMemory] = {}
        self.cold_storage: dict[str, str] = {}  # memory_id -> L0 text

    def add(self, raw_text: str, memory_id: str | None = None) -> CompactedMemory:
        """Add a new raw memory at L0."""
        if memory_id is None:
            memory_id = uuid.uuid4().hex[:8]
        now = time.time()
        memory = CompactedMemory(
            memory_id=memory_id,
            levels={CompactionLevel.L0_RAW: raw_text},
            entities=[],
            created_at=now,
            last_accessed=now,
        )
        self.memories[memory_id] = memory
        return memory

    def compact(self, memory_id: str, target_level: int) -> CompactedMemory:
        """Compact a memory to the target level. Archive L0 to cold storage."""
        memory = self.memories[memory_id]
        self.engine.compact(memory, target_level)

        # Move L0 to cold storage after any compaction
        if target_level >= CompactionLevel.L1_KEY_POINTS:
            if memory_id not in self.cold_storage:
                self.cold_storage[memory_id] = memory.levels[CompactionLevel.L0_RAW]

        return memory

The `retrieve` method is where multi-level access shines. You request a specific detail level. If it's already computed, you get it instantly. If the raw L0 was archived, it comes from cold storage. If the level hasn't been computed yet, the store triggers on-demand compaction.

In [ ]:
    def retrieve(self, memory_id: str, level: int | None = None) -> str:
        """Retrieve a memory at the requested detail level.

        If the level hasn't been computed yet, compaction runs on demand.
        If L0 is requested and has been archived, it comes from cold storage.
        """
        memory = self.memories[memory_id]
        memory.last_accessed = time.time()
        memory.access_count += 1

        if level is None:
            level = memory.current_level

        # Check if the level is already available
        if level in memory.levels:
            return memory.levels[level]

        # L0 might be in cold storage
        if level == CompactionLevel.L0_RAW and memory_id in self.cold_storage:
            return self.cold_storage[memory_id]

        # Need a level that hasn't been computed yet: compact on demand
        if level > memory.current_level:
            self.compact(memory_id, level)
            return memory.levels[level]

        return memory.levels[memory.current_level]

    def storage_report(self) -> dict:
        """Measure character counts at each compaction level."""
        totals = {level: 0 for level in CompactionLevel}
        for memory in self.memories.values():
            for level_key, content in memory.levels.items():
                totals[level_key] += len(content)
        cold_chars = sum(len(v) for v in self.cold_storage.values())
        return {
            "active_levels": {
                CompactionLevel(k).name: v for k, v in totals.items()
            },
            "cold_storage_chars": cold_chars,
            "total_memories": len(self.memories),
        }

### Compaction Scheduler

In a production system you don't compact every memory by hand. The scheduler checks which memories are due for further compression. It uses two policies:

- **Age-based**: compress memories older than a threshold (e.g., 24 hours).
- **Idle-based**: compress memories that haven't been accessed recently.

You can add more policies (storage-pressure, access-frequency) using the same pattern.

In [ ]:
class CompactionScheduler:
    """Selects memories for compaction based on age and access patterns."""

    def __init__(
        self,
        store: MemoryStore,
        age_threshold_seconds: float = 86400,   # 24 hours
        idle_threshold_seconds: float = 3600,    # 1 hour without access
    ):
        self.store = store
        self.age_threshold = age_threshold_seconds
        self.idle_threshold = idle_threshold_seconds

    def get_candidates(self) -> list:
        """Find memories eligible for further compaction.

        Returns a list of (memory_id, target_level) tuples.
        """
        now = time.time()
        candidates = []
        for mid, memory in self.store.memories.items():
            # Already at maximum compression
            if memory.current_level >= CompactionLevel.L3_TAGS:
                continue

            age = now - memory.created_at
            idle_time = now - memory.last_accessed
            next_level = memory.current_level + 1

            if age > self.age_threshold:
                candidates.append((mid, next_level))
            elif idle_time > self.idle_threshold:
                candidates.append((mid, next_level))

        return candidates

    def run(self) -> list:
        """Compact all eligible memories. Return list of compacted IDs."""
        candidates = self.get_candidates()
        compacted_ids = []
        for mid, target_level in candidates:
            self.store.compact(mid, target_level)
            compacted_ids.append(mid)
            print(f"  Compacted {mid} to {CompactionLevel(target_level).name}")
        return compacted_ids

## Example Run

Let's walk through a complete scenario. We'll add three realistic agent memories, compact them through all four levels, and see how storage shrinks at each step.

First, create the engine and store. Then add three raw memories that an agent might collect during a workday.

In [ ]:
engine = CompactionEngine(client=client, model=MODEL)
store = MemoryStore(engine=engine)

# Memory 1: A CI/CD planning conversation
memory_1 = store.add(
    memory_id="cicd-plan",
    raw_text=(
        "User: We need to set up CI/CD for the new microservices project. "
        "We're using Kubernetes on AWS EKS. The team uses GitHub for version control. "
        "Assistant: I'll help you set up CI/CD for your Kubernetes microservices on "
        "AWS EKS. I recommend GitHub Actions for the pipeline. Each service gets its "
        "own workflow file. The pipeline should build a Docker image, push it to ECR, "
        "then deploy via Helm charts. User: Sounds good. What about staging vs "
        "production? Assistant: Use separate EKS namespaces for staging and production. "
        "The staging deploy triggers on every push to main. Production requires a "
        "manual approval step in the GitHub Actions workflow. User: What about "
        "rollbacks? Assistant: Helm tracks release history. Run 'helm rollback' to "
        "revert to any previous version. Set up health checks so Kubernetes "
        "auto-restarts unhealthy pods. User: Great, let's go with that plan."
    ),
)

# Memory 2: A debugging session
memory_2 = store.add(
    memory_id="debug-500",
    raw_text=(
        "User: The checkout service is throwing 500 errors since yesterday's deploy. "
        "About 15% of requests fail. Assistant: Let me help debug that. What changed "
        "in yesterday's deploy? User: We updated the payment gateway SDK from v2.3 to "
        "v3.0 and added a new discount calculation module. Assistant: The payment SDK "
        "jump from v2 to v3 is likely the cause. v3 changed the response format from "
        "XML to JSON. Your deserialization code probably expects XML. Check the "
        "PaymentResponseParser class. User: You're right. The parser was calling "
        "xml.etree.parse on JSON responses. I've switched it to json.loads and the "
        "errors stopped. Assistant: Good catch. I'd also recommend adding a response "
        "format check so future SDK updates don't break silently. Log the content-type "
        "header before parsing."
    ),

We add a third memory: a product roadmap discussion. After loading all three, we check their character counts at L0.

In [ ]:

# Memory 3: A product roadmap discussion
memory_3 = store.add(
    memory_id="q2-roadmap",
    raw_text=(
        "User: We need to plan the Q2 product roadmap. Main themes are mobile app "
        "improvements and API v2 launch. The team has 12 engineers. Assistant: Let's "
        "break Q2 into two tracks. Track A: Mobile improvements with 5 engineers. "
        "Track B: API v2 with 7 engineers. User: For mobile, the priorities are "
        "offline mode, push notifications, and biometric login. Assistant: I'd suggest "
        "this order: biometric login first (2 weeks, unblocks other features), then "
        "push notifications (3 weeks, high user demand), then offline mode (5 weeks, "
        "most complex). User: For API v2, we need rate limiting, webhook support, and "
        "OAuth2 scopes. Assistant: Start with OAuth2 scopes since other features "
        "depend on the auth model. Then rate limiting (protects the new endpoints). "
        "Webhooks last because they need the stable endpoint structure. User: Timeline "
        "works. Let's write this up and share with the team."
    ),
)

print(f"Added {len(store.memories)} memories to the store.")
for mid, mem in store.memories.items():
    chars = mem.char_count_at(CompactionLevel.L0_RAW)
    print(f"  {mid}: {chars} characters at L0")

### Step-by-Step Compaction

Let's compact the first memory one level at a time. You'll see exactly what the LLM produces at each stage.

In [ ]:
# Compact cicd-plan from L0 to L1 (key points)
store.compact("cicd-plan", CompactionLevel.L1_KEY_POINTS)
mem = store.memories["cicd-plan"]

print("=== L0: Raw transcript ===")
print(mem.levels[CompactionLevel.L0_RAW][:200] + "...")
print(f"\n({mem.char_count_at(CompactionLevel.L0_RAW)} chars)")

print("\n=== L1: Key points ===")
print(mem.levels[CompactionLevel.L1_KEY_POINTS])
print(f"\n({mem.char_count_at(CompactionLevel.L1_KEY_POINTS)} chars)")

ratio = mem.char_count_at(CompactionLevel.L1_KEY_POINTS) / mem.char_count_at(CompactionLevel.L0_RAW)
print(f"\nCompression ratio: {ratio:.0%} of original")

Now compress from L1 to L2 (single sentence), then from L2 to L3 (tags only).

In [ ]:
# L1 -> L2 -> L3
store.compact("cicd-plan", CompactionLevel.L3_TAGS)
mem = store.memories["cicd-plan"]

print("=== L2: One-sentence summary ===")
print(mem.levels[CompactionLevel.L2_SUMMARY])
print(f"({mem.char_count_at(CompactionLevel.L2_SUMMARY)} chars)")

print("\n=== L3: Tags only ===")
print(mem.levels[CompactionLevel.L3_TAGS])
print(f"({mem.char_count_at(CompactionLevel.L3_TAGS)} chars)")

print(f"\nExtracted entities: {mem.entities}")

# Show the full compression ladder
print("\n=== Compression ladder ===")
for level in CompactionLevel:
    if level in mem.levels:
        chars = len(mem.levels[level])
        pct = chars / mem.char_count_at(CompactionLevel.L0_RAW) * 100
        print(f"  {level.name}: {chars} chars ({pct:.1f}% of L0)")

### Batch Compaction

Compact the remaining two memories to L3 in a single call each. The engine runs all intermediate steps automatically.

In [ ]:
for mid in ["debug-500", "q2-roadmap"]:
    store.compact(mid, CompactionLevel.L3_TAGS)
    mem = store.memories[mid]
    l0_chars = mem.char_count_at(CompactionLevel.L0_RAW)
    l3_chars = mem.char_count_at(CompactionLevel.L3_TAGS)
    print(f"{mid}:")
    print(f"  L0: {l0_chars} chars")
    print(f"  L3: {l3_chars} chars ({l3_chars / l0_chars:.1%} of L0)")
    print(f"  Entities: {mem.entities}")
    print()

### Level-Aware Retrieval

The key advantage of multi-level storage: you pick the right resolution for each query. A broad search scans L3 tags. A focused lookup reads L1 key points. A full audit retrieves L0 from cold storage.

In [ ]:
print("--- Retrieving 'debug-500' at different levels ---\n")

for level in CompactionLevel:
    content = store.retrieve("debug-500", level=level)
    preview = content[:120].replace("\n", " ")
    if len(content) > 120:
        preview += "..."
    print(f"{level.name}:")
    print(f"  {preview}")
    print(f"  ({len(content)} chars)\n")

### Storage Savings

Let's measure how much space compaction saves across all three memories.

In [ ]:
report = store.storage_report()

print(f"Total memories: {report['total_memories']}")
print(f"\nActive store by level:")
for level_name, chars in report["active_levels"].items():
    print(f"  {level_name}: {chars:,} chars")
print(f"\nCold storage (archived L0): {report['cold_storage_chars']:,} chars")

# Calculate overall compression
total_l0 = sum(
    mem.char_count_at(CompactionLevel.L0_RAW)
    for mem in store.memories.values()
)
total_l3 = sum(
    mem.char_count_at(CompactionLevel.L3_TAGS)
    for mem in store.memories.values()
)
print(f"\nTotal L0 content: {total_l0:,} chars")
print(f"Total L3 content: {total_l3:,} chars")
print(f"Compression ratio (L3/L0): {total_l3 / total_l0:.1%}")
print(f"\nIf you retrieve at L3, you use ~{total_l3 / total_l0:.0%} of the tokens.")
print(f"The original text stays safe in cold storage for when you need it.")

### Scheduled Compaction

In a real system, you don't compact by hand. The scheduler runs periodically and compresses memories that meet the policy criteria. Let's demonstrate with a short idle threshold.

In [ ]:
# Create a fresh store for the scheduling demo
sched_engine = CompactionEngine(client=client, model=MODEL)
sched_store = MemoryStore(engine=sched_engine)

# Add two memories
sched_store.add(
    memory_id="old-mem",
    raw_text="User asked about Python virtual environments. We discussed venv, "
    "conda, and poetry. Recommended poetry for dependency management in "
    "production projects. Covered lock files and reproducible builds.",
)

sched_store.add(
    memory_id="new-mem",
    raw_text="User wants to learn about async programming in Python. Discussed "
    "asyncio event loops, coroutines, and the await keyword. Recommended "
    "starting with simple HTTP requests using aiohttp.",
)

# Backdate the first memory so it looks old
sched_store.memories["old-mem"].created_at -= 90000   # ~25 hours ago
sched_store.memories["old-mem"].last_accessed -= 90000

# Create scheduler with 24-hour age threshold
scheduler = CompactionScheduler(
    store=sched_store,
    age_threshold_seconds=86400,   # 24 hours
    idle_threshold_seconds=3600,   # 1 hour
)

print("Before scheduling:")
for mid, mem in sched_store.memories.items():
    print(f"  {mid}: level={CompactionLevel(mem.current_level).name}")

print("\nRunning scheduler...")
compacted = scheduler.run()

print(f"\nCompacted {len(compacted)} memories.")
print("\nAfter scheduling:")
for mid, mem in sched_store.memories.items():
    print(f"  {mid}: level={CompactionLevel(mem.current_level).name}")

## Tradeoffs

### When Memory Compaction Works Well

- **Long-running agents** that accumulate thousands of raw memories over days or weeks. Compaction keeps the store searchable without blowing up storage costs.
- **Token-constrained systems** where retrieved context must fit within tight budget limits. L2 and L3 summaries let you pack more memories into a single prompt.
- **Multi-resolution retrieval**: quick tag-based filtering at L3, then expanding top results to L1 for richer context. This two-pass pattern is fast and cost-effective.

### When It Breaks Down

- **Latency-sensitive writes.** Each compaction step requires an LLM call. If you need instant writes, you'll need to batch compaction into background jobs. That adds operational complexity.
- **Exact-recall tasks.** Summarization is lossy. If the agent later needs the precise wording from turn 47, the L2 summary won't have it. Cold storage retrieval helps, but adds latency.
- **Compaction cost.** Each memory requires up to 3 LLM calls to reach L3. For high-throughput agents producing hundreds of memories per hour, the compaction cost may rival the storage savings.
- **Small memory stores.** If your agent has fewer than 100 memories, the overhead of multi-level compaction outweighs the benefit. Buffer or window memory is a better fit.

## Further Reading

- Forte, ["Progressive Summarization: A Practical Technique for Designing Discoverable Notes,"](https://fortelabs.com/blog/progressive-summarization-a-practical-technique-for-designing-discoverable-notes/?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques) 2017. The original progressive summarization method for personal knowledge management. Directly inspires the multi-level compaction approach.
- Xu et al., ["RECOMP: Improving Retrieval-Augmented LMs with Compression and Selective Augmentation,"](https://arxiv.org/abs/2310.04408) 2023. Compresses retrieved documents before injecting them into context. Shows that compressed content can match or exceed raw content performance.
- Chevalier et al., ["Adapting Language Models to Compress Contexts,"](https://arxiv.org/abs/2305.14788) 2023. Trains models to produce compact context representations. A learned approach to memory compaction.
- Jiang et al., ["LongLLMLingua: Accelerating and Enhancing LLMs in Long Context Scenarios via Prompt Compression,"](https://arxiv.org/abs/2310.06839) 2023. Prompt compression techniques that reduce token count while preserving task performance.

---

*← Previous: [14 - Memory Consolidation](../14_memory_consolidation/) · Next: [16 - Self-Reflection Memory](../16_self_reflection_memory/) →*

## 🧪 Try It Yourself

Three small challenges to deepen your understanding. Each should take 10-30 minutes.

### Challenge 1: Custom intermediate level
Add a new `L1_5_HIGHLIGHTS` level to `CompactionLevel` between L1 (key points) and L2 (one-liner). Write a `compact_to_l1_5()` method in `CompactionEngine` that extracts only the top 2 most important points. Test it on 5 memories and compare output length against L1 and L2.

### Challenge 2: Compression ratio analysis
Compact 10 memories through all four levels. For each memory, measure the token count at L0, L1, L2, and L3. Compute the compression ratio (L0 tokens / LN tokens) at each level. Plot a bar chart showing average compression ratios across the 10 memories.

### Challenge 3: Level-aware retrieval
Modify `MemoryStore.retrieve()` to search L2 representations first (fast, cheap) and then fetch the L0 or L1 version of the top results (detailed, accurate). Measure retrieval quality and token cost against always searching L0. This connects to the hybrid retrieval ideas in 20 Memory Retrieval Patterns.


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--15-memory-compaction--memory-compaction)